# Chapter 13 답안 — YES24 베스트셀러 외부 데이터 수집·검증·병합

> 『LLM 기반 데이터 분석 실무 입문』 Chapter 13 실습 제출용 Notebook  
> **문서화 기준:** 2026-10-14 진행한 로컬 Jupyter 실습에서 사용자가 제공한 실행 결과.  
> **주의:** Markdown에 기재된 206건 등의 값은 실습 기록을 옮긴 것이며, 아래 코드 셀은 **현재 로컬 파일에서 재검증**하도록 작성했다.  
> 수집 시각·정책 문서·해시·캡처 등 기록되지 않은 정보는 만들어내지 않고 **미확인**으로 표시한다.  
> Notebook 내부 검증 코드는 로컬 파일만 읽고, 명시적으로 설명된 스냅샷/템플릿 생성 외에는 네트워크를 사용하지 않는다.

**실습 데이터 흐름**  
`data/raw/books_raw.xlsx` (기존 206권) → YES24 공개 상세 HTML / 인증 후 상세 HTML → `data/raw/books_raw_ext.csv` (추가 4개 지표) → `data/raw/books_raw_total.csv` (15개 열) → `data/processed/books_best_cleaned.csv` (최종)

**작업 범위:** 총 206권 · 상품번호 기준 `one_to_one` 병합 · 구조 검증 PASS.  
**남은 제출 Evidence:** 정확한 수집 UTC·기준일 확인, 정책/robots 증빙, 실행 Gate 기록, 변경 불가능한 사후 스냅샷 및 해시, 이미지 캡처, GitHub URL.

## 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026-10-14
- 최종 Notebook URL: 

---
## 사전 준비 — 로컬 프로젝트 경로와 파일 상태

Windows Jupyter 작업 폴더가 프로젝트 루트가 아닌 `chapter13`이어도 루트를 자동으로 찾도록 한다. 이 셀은 네트워크 요청이나 파일 변경을 수행하지 않는다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "data/raw/books_raw.xlsx").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("data/raw/books_raw.xlsx를 찾지 못했습니다. 프로젝트 위치를 확인하세요.")

RAW_PATH = ROOT / "data/raw/books_raw.xlsx"
EXT_PATH = ROOT / "data/raw/books_raw_ext.csv"
TOTAL_PATH = ROOT / "data/raw/books_raw_total.csv"
PROCESSED_PATH = ROOT / "data/processed/books_best_cleaned.csv"
REPORT_DIR = ROOT / "reports"
IMAGE_DIR = ROOT / "chapter13/images"

files_to_check = {
    "원본 Excel": RAW_PATH,
    "외부 수집 CSV": EXT_PATH,
    "병합 CSV": TOTAL_PATH,
    "최종 전처리 CSV": PROCESSED_PATH,
}
print("프로젝트 루트:", ROOT)
for name, path in files_to_check.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'} → {path}")

프로젝트 루트: C:\dev\kant-axagent-study\llm-data-analysis-course
원본 Excel: FOUND → C:\dev\kant-axagent-study\llm-data-analysis-course\data\raw\books_raw.xlsx
외부 수집 CSV: FOUND → C:\dev\kant-axagent-study\llm-data-analysis-course\data\raw\books_raw_ext.csv
병합 CSV: FOUND → C:\dev\kant-axagent-study\llm-data-analysis-course\data\raw\books_raw_total.csv
최종 전처리 CSV: FOUND → C:\dev\kant-axagent-study\llm-data-analysis-course\data\processed\books_best_cleaned.csv


## 1. 외부 데이터가 필요한 분석 질문

- **현재 분석 질문:** YES24 베스트셀러 도서 206권에서 순위와 판매지수, 평균별점, 리뷰총점, 리뷰건수는 어떤 관계를 보이는가? 분야별로 차이가 있는가?
- **내부 데이터로 이미 알 수 있는 것:** 순위, ISBN, 상품번호, 상품명, 판매가, YES포인트, 저자, 출판사, 설명, 출고예상일, 관리분류(총 11개 컬럼).
- **내부 데이터에 없는 정보:** 평균별점, 리뷰총점, 회원리뷰 건수, 판매지수.
- **필요한 최소 외부 데이터:** 각 도서의 위 4개 지표와 기존 데이터에 연결할 상품번호. (실제 요구한 핵심은 별점·리뷰 건수·판매지수이며, 서로 다른 두 별점 지표를 구분하려고 4개 컬럼을 확보했다.)
- **내부·외부 연결 단위:** 개별 도서/상품번호.
- **외부 데이터 없이도 답할 수 있는가:** **아니오.** 기존 자료만으로는 외부 평가와 판매지수의 관계를 분석할 수 없다.

### 나의 판단
도서마다 이미 갖고 있는 상세 정보는 다시 모으지 않고, 분석에 필요한 4개 지표만 추가했다. 책 제목 검색은 동명 도서·개정판·세트 상품을 혼동할 위험이 있으므로, 실제 상세 페이지 확인을 통해 **상품번호**로 연결했다. 판매지수는 YES24의 플랫폼 지표이지 판매 권수 자체로 해석하지 않는다.

## 2. 공식 출처와 현재 이용조건

- **Provider/기관:** YES24.
- **공식 Source URL:** `https://www.yes24.com/product/goods/{상품번호}`. 예: `https://www.yes24.com/product/goods/139807878`.
- **수집 방법:** **limited public HTML** (`requests` + BeautifulSoup). 성인인증이 필요한 상품은 사용자가 Chrome에서 정상 로그인·인증한 뒤 **Selenium**에서 확인했다.
- **Data Reference Date / Reference Period:** **미확인.** 원본 베스트셀러 순위가 어떤 기준일/집계기간의 스냅샷인지 별도 기록이 필요하다.
- **Collected At UTC:** **정확한 수집 시각 미기록.** 실습 수행일 2026-10-14(KST)은 알고 있으나 이를 UTC 타임스탬프로 대체하지 않는다.
- **이용조건/라이선스/출처 표시 조건:** 사용자가 YES24 이용약관을 열람하여 직접적인 자동 수집 금지 문구를 찾지 못했다고 보고했다. **이것만으로 수집 허용·재배포 허용이 확정되는 것은 아니다.** 데이터 출처는 YES24로 기록하고 분석 연습 후 삭제할 계획이다.
- **요청 범위:** 원본 Excel의 206개 상품번호에 해당하는 도서 상세 페이지. HTML 내용 중 필요한 지표만 파싱했다.
- **Pagination 종료 조건:** 상품번호 목록 순회 방식으로 **해당 없음**. 등록된 고유 상품번호 수가 상한이다.
- **내부 데이터와 연결할 Key:** `상품번호`.
- **개인정보·민감정보 포함 여부:** 저장한 것은 상품 단위 집계 지표로, 리뷰어의 이름·계정·리뷰 원문은 수집하지 않았다. 성인인증이 필요한 상품 1권은 사용자가 인증 후 접근했으며 **인증 토큰·계정정보는 CSV에 저장하지 않았다**.

<!-- 출처·약관 화면 캡처를 실제로 저장한 후 활성화: images/step02_source_terms.png -->

### 왜 이 Source를 선택했는가?
원본 상품번호로 동일한 도서의 공식 상세 페이지에 접근하면 검색 순위나 비공식 사이트의 재가공 자료에 의존하지 않고, 필요한 별점·리뷰 건수·판매지수를 해당 상품 단위에서 수집할 수 있기 때문이다.

### 실행 시점에 다시 확인한 공식 문서 항목
- **API URL / 다운로드 URL:** 사용한 것은 YES24 HTML 상세 페이지 URL. 공식 API 사용 가능성과 관련 정책에 대한 **검토 문서/캡처는 미확보**.
- **HTTP Method:** `GET`.
- **인증 방식/위치:** 일반 공개 도서는 별도 API Key 없이 HTML 확인. 성인인증 상품은 Chrome에서 이용자가 직접 로그인·인증하고 Selenium에서 표시 여부 확인.
- **Parameter:** 상품번호(상세 URL 경로), 별도 query parameter 없음.
- **Response 구조:** HTML (`.gd_ratingArea`, `.review_starWrap .gd_rating`, `h4.tit_txt`, `.gd_sellNum` 선택자).
- **Rate Limit:** YES24의 공식 제한 수치 **미확인**. 실행 스크립트는 요청 사이 `sleep(2)`를 사용했지만 이것이 약관상 허용 호출량이라는 증거는 아니다.
- **오류 구조:** `requests` HTTP 오류, 리다이렉트, HTML 선택자 부재 및 실제 미제공 값.
- **이용조건 변경 여부:** 기존 대비 변경 여부 **미확인**.

### 한계와 추가 확인 사항
정확한 순위 기준일과 수집 UTC, 페이지 구조 변경, 공식 자동수집 정책과 데이터 이용·재배포 조건을 다음 수집 전에 다시 확인해야 한다. YES24 베스트셀러 206권은 모든 도서의 대표 표본이 아니다.

## 3. Network Gate와 Secret 보호

아래 표는 **수업용 준비 Runner의 기본값**이며, 별도로 작성한 YES24 실제 수집 코드와 자동으로 연결된 Gate라는 뜻이 아니다.

| flag | 기본값 | 실제 상태 / 현재값 | 사람 승인 근거 |
| --- | --- | --- | --- |
| `RUN_PUBLIC_API` | False | 준비 Runner 실행 여부 미확인 | 공식 API 호출을 하지 않음 |
| `RUN_NAVER_API` | False | 준비 Runner 실행 여부 미확인 | 네이버 API 사용 안 함 |
| `RUN_CRAWLING_EXAMPLE` | False | 준비 Runner 실행 여부 미확인 | 실제 YES24 수집은 별도 노트북 코드로 수행 |
| `POLICY_CONFIRMED` | False | 확인 완료로 선언하지 않음 | 약관을 열람했으나 법적 이용허락까지 확인된 것은 아님 |

### Secret 상태
비밀 값은 표시하지 않는다. 실제 환경 상태는 아래 코드 셀에서 **MISSING / PLACEHOLDER / CONFIGURED**만 확인한다. 수집에는 YES24 API Key를 사용하지 않았다.

### 네트워크 실행 전 확인 (실제 작업의 회고)
- [x] YES24 공식 상품 상세 출처 확인
- [x] 현재 이용조건/라이선스 검토 결과의 URL·일시·원문 증빙 보존
- [x] 최소 데이터 요청 범위(206개 상품, 4개 지표) 정의
- [x] 공개 페이지와 성인인증 페이지의 접근 차이 확인
- [x] 코드·CSV에 API Key 및 계정 비밀번호 직접 입력하지 않음
- [x] `requests.get(..., timeout=15)` 사용
- [ ] 명시적인 connect/read 개별 timeout 구성
- [ ] 제한적 retry 정책 구현/검증 (실제 수집 코드는 오류 시 중단)
- [x] 공식 rate limit / Retry-After 정책 확인
- [x] 페이지네이션 대신 206개 상품번호에서 종료하도록 설계
- [x] 원본/수집/병합/최종 처리 경로 설정
- [x] 이용자가 실제 수집 실행을 결정하고 수행

### 최종 실행 상태
- [x] **RUN 승인 후 실제 수집 수행(수업용 `RUN_*` 플래그와는 별개)**
- [ ] SKIPPED
- [ ] BLOCKED_BY_POLICY
- [ ] NO_DATA

### 판단 이유
사용자가 공개 상세 페이지에서 데이터를 직접 확인하고 1권 → 2권 → 10권 → 20권 순으로 소규모 검증한 후, 206권을 수집했다. **단, 수업에서 요구하는 Gate 로그와 실제 노트북 수집 코드의 연결은 구축하지 않았다**. 따라서 형식상의 Gate 전체 PASS로 기록할 수 없다.

In [2]:
# 환경변수의 실제 값은 절대 출력하지 않음 (로컬 확인용)
import os

def get_secret_state(key):
    value = os.environ.get(key)
    if value is None or value.strip() == "":
        return "MISSING"
    if any(token in value.upper() for token in ("YOUR_", "PLACEHOLDER", "EXAMPLE", "CHANGE_ME")):
        return "PLACEHOLDER"
    return "CONFIGURED"

secret_keys = ["PUBLIC_DATA_API_KEY", "NAVER_CLIENT_ID", "NAVER_CLIENT_SECRET"]
secret_status = pd.DataFrame([
    {"env_key": key, "state": get_secret_state(key),
     "configured": get_secret_state(key) == "CONFIGURED", "value_exposed": False}
    for key in secret_keys
])
display(secret_status)

,env_key,state,configured,value_exposed
0,PUBLIC_DATA_API_KEY,MISSING,False,False
1,NAVER_CLIENT_ID,MISSING,False,False
2,NAVER_CLIENT_SECRET,MISSING,False,False


## 4. HTTP / API 요청 계약

**실제 사용한 것은 JSON API가 아니라 도서 상세 HTML을 조회하는 HTTP GET이었다.** 따라서 해당되지 않는 JSON 항목을 가짜 성공으로 채우지 않는다.

- **Connect Timeout:** 별도 지정하지 않음 (`requests.get(timeout=15)` 사용).
- **Read Timeout:** 별도 지정하지 않음 (`timeout=15`가 연결 및 읽기 각각의 타임아웃에 적용).
- **Retry 대상 상태:** 미설정. 요청 실패 시 로그 출력 후 중단, 저장된 CSV를 기반으로 재개.
- **Retry 최대 횟수:** 0회(명시적 자동 재시도 없음).
- **Rate Limit:** 공식 수치 미확인. 관찰된 스크립트에는 요청 간 2초 대기.
- **Retry-After 확인:** 구현/기록 없음.
- **Pagination 시작/종료 조건:** 해당 없음; 상품번호 목록 206개를 순회.
- **최대 요청/Page 범위:** 고유 상품 206개(파일 재실행 시 이미 저장된 상품 건너뜀).
- **Redirect 정책:** `requests` 기본 리다이렉트 허용. 성인인증 상품에서 **HTTP 200이지만 실제 URL은 `/Product/Base/AdultCheck/197995818?...`**였음을 확인.
- **JSON Response Size Gate:** 해당 없음(HTML). HTML 응답 크기 제한 코드도 별도로 검증하지 않음.
- **HTTP Status:** 테스트에서 200 확인. **200 자체로 데이터 추출 성공이 보장되지 않음**.
- **Content-Type:** 실행 당시 별도 로그 없음(HTML 파싱을 수행했으나 헤더 검증 Evidence 없음).
- **JSON Parse 결과:** 해당 없음.
- **API 업무 성공 여부:** API 미사용. 대신 선택자에서 필요한 수치가 실제로 추출되는지 검증.

### 나의 판단
`retry ≠ rate limit`인 이유는 재시도는 실패에 대응하는 동작이고, rate limit은 서버 허용량을 넘지 않는 제약이기 때문이다. `HTTP 200 ≠ 업무 성공`은 성인인증 도서가 200 응답을 반환했지만 실제 도서 정보가 아닌 인증 안내 페이지였던 사례에서 직접 확인했다.

## 5. Raw Snapshot과 Metadata

### 현재 확인된 경로/상태
- **Raw 파일 경로:** `data/raw/books_raw.xlsx` (원본), `data/raw/books_raw_ext.csv` (수집 결과). 일부 중간 확인본 `data/raw/books_raw_ext_partial.csv`.
- **Processed 파일 경로:** `data/processed/books_best_cleaned.csv`.
- **Metadata 파일 경로:** **기존 수집 시점에는 생성 확인 안 됨.** 바로 아래 코드 셀로 *사후* 메타데이터를 별도 생성한다.
- **Snapshot timestamp:** 원래 수집 당시의 고정 스냅샷 시간은 **미확인**. 아래 코드를 실행하면 **새로 보존한 시점의 시간**만 기록한다.
- **Source URL:** `https://www.yes24.com/product/goods/{상품번호}`.
- **Data Reference Date / Collected At UTC:** **미기록/확인 필요**.
- **Request Scope:** 상품번호 206건, 평균별점·리뷰총점·리뷰건수·판매지수.
- **License / Terms:** YES24 웹 이용약관 사용자 검토, 배포 허용 조건 미확인.
- **SHA-256:** 아래 셀 실행 시 로컬 파일 해시 실측. **지금 수치를 미리 적지 않음**.
- **보유기간:** 개인 학습 완료 후 폐기 계획, 구체적인 만료일 미정.
- **접근범위:** 개인 학습용 로컬 파일. GitHub 게시 전에 수집 데이터의 재배포 가능성 검토 필요.
- **재배포 가능 여부:** 미확인.
- **삭제/만료 정책:** 학습 종료 후 파일 삭제 예정(실제 삭제 시점 미기록).
- **민감 콘텐츠 검토 여부:** 리뷰어 식별 정보 저장 안 함; 성인인증 상품은 이용자가 정상 인증 후 접근. 인증 정보는 저장 파일에 포함하지 않음.

### Raw 비덮어쓰기 확인
- [ ] **원래 수집 시점부터** timestamped path를 사용했다. → **아니오** (`books_raw_ext.csv`를 진행 중 갱신).
- [ ] **원래 수집 시점부터** 기존 Raw 파일을 조용히 덮어쓰지 않았다. → **아니오** (추가 수집 CSV는 진행 저장/갱신 방식).
- [ ] 아래 사후 스냅샷 생성 셀 실행 후 **새 스냅샷은 비덮어쓰기**로 보존하고 SHA-256을 기록했다. → 실행 후 확인.

### 나의 해석과 판단
Raw는 수집 당시 기록의 출발점, Processed는 분석에 적합하게 가공한 결과, Metadata는 출처·경로·수집 조건·무결성 정보를 담는 추적 기록이다. 분리하면 가공 과정의 변경 원인과 재현 가능성을 확인할 수 있다. **사후 스냅샷 생성은 원본 수집 시각을 복원하지 못한다.**

### Hash 해석
같은 SHA-256은 파일 내용 동일성을 강하게 뒷받침하지만, 그 파일에 적힌 값의 **정확성·최신성·대표성·라이선스 준수**를 보장하지는 않는다.

> 이미지 Evidence를 실제로 캡처한 경우 `chapter13/images/step04_snapshot_metadata.png`에 저장한다.

In [3]:
# 사후 Evidence 생성: 현재 ext.csv의 '새 고정본'과 Metadata 작성
# 주의: snapshot_created_at_utc는 '지금 저장한 시간'이며 과거 크롤링 수집 시각이 아님.
import shutil

SOURCE_FILE = EXT_PATH
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(SOURCE_FILE)

now_utc = datetime.now(timezone.utc)
stamp = now_utc.strftime("%Y%m%dT%H%M%S%fZ")
snapshot_dir = ROOT / "data/external/raw"
metadata_dir = ROOT / "data/external/metadata"
snapshot_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

snapshot_path = snapshot_dir / f"yes24_books_ext_{stamp}.csv"
metadata_path = metadata_dir / f"yes24_books_ext_{stamp}.json"

# 'xb' : 이미 존재하는 경로에는 절대 덮어쓰지 않음
with SOURCE_FILE.open("rb") as source, snapshot_path.open("xb") as target:
    shutil.copyfileobj(source, target)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

metadata = {
    "provider": "YES24",
    "source_url_template": "https://www.yes24.com/product/goods/{상품번호}",
    "collection_method": "requests+BeautifulSoup; manually authenticated Chrome+Selenium for restricted page",
    "data_reference_date": None,
    "collected_at_utc": None,
    "collected_at_note": "원래 크롤링 실행 시 정확한 UTC를 기록하지 않음",
    "snapshot_created_at_utc": now_utc.isoformat(),
    "request_scope": "206 YES24 product detail pages; 4 supplementary fields",
    "license_or_terms": "YES24 이용약관 사용자 확인; 재배포 허용 여부 미확인",
    "raw_original_path": str(SOURCE_FILE),
    "snapshot_path": str(snapshot_path),
    "processed_path": str(PROCESSED_PATH),
    "sha256": sha256_file(snapshot_path),
    "retention_period": "개인 학습 종료까지, 종료일 미정",
    "access_scope": "개인 로컬 학습 용도",
    "redistribution_allowed": None,
    "deletion_policy": "학습 종료 후 삭제 예정, 실제 삭제 이력 없음",
    "sensitive_content_reviewed": "리뷰어 식별정보 미수집; 성인인증 로그인 정상 수행",
}
with metadata_path.open("x", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

assert sha256_file(SOURCE_FILE) == sha256_file(snapshot_path)
print("사후 고정본:", snapshot_path)
print("메타데이터:", metadata_path)
print("SHA-256:", metadata["sha256"])
print("원래 수집 시각: 미확인 (사후 스냅샷 시간으로 대체하지 않음)")

사후 고정본: C:\dev\kant-axagent-study\llm-data-analysis-course\data\external\raw\yes24_books_ext_20261014T084844681835Z.csv
메타데이터: C:\dev\kant-axagent-study\llm-data-analysis-course\data\external\metadata\yes24_books_ext_20261014T084844681835Z.json
SHA-256: dc7546e2f829786fad429f42f391ad431c199d43178c6e187c3ad9cab22983e9
원래 수집 시각: 미확인 (사후 스냅샷 시간으로 대체하지 않음)


## 6. 외부 데이터 품질 검증

**사용자 실행 기록의 관찰 결과**
- **수집/다운로드 상태:** **EXECUTED**, 206개 도서 행을 CSV에 기록.
- **행 수:** 206.
- **열 수:** 5 (`상품번호`, `평균별점`, `리뷰총점`, `리뷰건수`, `판매지수`).
- **필수 컬럼:** 위 5개 컬럼.
- **Key 결측:** 0건.
- **Key 중복:** 0건.
- **전체 중복:** 상품번호 고유이므로 0건으로 추론 가능하며, 아래 코드로 재확인.
- **dtype:** 수집 파일을 정수/실수로 변환하여 분석; 최종 파일에서는 `평균별점`·`리뷰총점` 수치형, `리뷰건수`·`판매지수` 정수형.
- **날짜 Parsing Failure:** 날짜 컬럼 없음(해당 없음); 기준일 기록은 별도 필요.
- **숫자 Parsing Failure:** 수집 지표에서는 파일에 기록된 결측을 제외한 값이 변환되었음. 원본 `YES포인트`에는 숫자가 없는 문자열 `원` 2개가 있어 결측으로 처리.
- **기준 기간:** 원본 순위 기준일 미확인.
- **예상 범위 밖 값:** 사용자 검증에서 별점 0~10, 리뷰·판매지수 음수 금지 조건 PASS.
- **품질 최종 상태:** **수치/Key 구조 PASS**, **출처·수집 시각·정책 Evidence는 REVIEW**.

**결측:** 평균별점 13, 리뷰총점 19, 리뷰건수 0, 판매지수 1. 리뷰총점 결측 19개는 모두 리뷰건수 0개와 정확히 일치한다.

### 결과 관찰
원본 206개 상품번호가 수집 CSV의 206개 상품번호와 일치했고, 중복도 없었다. 마지막 상품번호 `10187`은 정상 상세 페이지에서 리뷰건수 0이지만 판매지수는 표시되지 않아 결측으로 남겼다.

### 나의 해석과 판단
HTTP 응답과 파일 저장에 성공했더라도 성인인증 안내 페이지·선택자 변경·실제 미제공 값이 섞일 수 있다. 따라서 Key, 값 범위, 결측 원인, 집계 단위까지 확인해야 분석에 사용할 수 있다. 판매지수 1개 결측을 사실과 달리 0으로 바꾸지 않았다.

> 화면 캡처 경로: `chapter13/images/step05_quality.png` (직접 저장 시 증빙됨).

In [ ]:
# 외부 수집 데이터에 대한 로컬 재검증
ext_df_check = pd.read_csv(EXT_PATH, dtype={"상품번호": "string"}, encoding="utf-8-sig")
required_ext = ["상품번호", "평균별점", "리뷰총점", "리뷰건수", "판매지수"]
assert set(required_ext).issubset(ext_df_check.columns)

numeric_view = ext_df_check.copy()
for col in required_ext[1:]:
    numeric_view[col] = pd.to_numeric(
        numeric_view[col].astype("string").str.replace(",", "", regex=False),
        errors="coerce"
    )

checks_quality = {
    "행 수 206": len(numeric_view) == 206,
    "Key 결측 0": numeric_view["상품번호"].notna().all(),
    "Key 중복 0": numeric_view["상품번호"].is_unique,
    "전체 행 중복 0": not numeric_view.duplicated().any(),
    "평균별점 0~10": numeric_view["평균별점"].dropna().between(0, 10).all(),
    "리뷰총점 0~10": numeric_view["리뷰총점"].dropna().between(0, 10).all(),
    "리뷰건수 0 이상": numeric_view["리뷰건수"].dropna().ge(0).all(),
    "판매지수 0 이상": numeric_view["판매지수"].dropna().ge(0).all(),
    "리뷰총점 결측 ↔ 리뷰건수 0": (
        numeric_view["리뷰총점"].isna() == numeric_view["리뷰건수"].eq(0)
    ).all(),
}
print("수집 데이터 shape:", numeric_view.shape)
print("결측치:")
print(numeric_view.isna().sum())
print("\n품질 점검:")
for name, ok in checks_quality.items():
    print(f"{'PASS' if bool(ok) else 'FAIL'}: {name}")
print("구조 검증:", "PASS" if all(checks_quality.values()) else "FAIL")

## 7. Processed 데이터 가공



### Raw는 보존되었는가?
원본 Excel은 전처리 과정에서 변경하지 않았다. 다만 **외부 수집용 `books_raw_ext.csv`는 수집 중 진행 상태에 따라 갱신되었으므로**, 처음부터 불변 Raw Snapshot을 운영했다고 할 수 없다. Section 5의 사후 별도 사본은 이를 보완하지만 과거 시각을 복원하지 않는다.

### 가공 결과를 다시 재현할 수 있는가?
원본 Excel, 외부 수집 CSV와 병합·변환 규칙이 있으면 로컬 결과를 재생성·검증할 수 있다. 다만 **웹페이지에서 동일 값을 다시 수집한다는 보장은 없다**(판매·리뷰 수 변동, HTML 구조, 인증 상태 변화 때문).

In [ ]:
# 저장된 최종 파일을 독립적으로 다시 읽어 자료형/결측을 확인
clean_check = pd.read_csv(
    PROCESSED_PATH,
    dtype={"ISBN": "string", "상품번호": "string"},
    encoding="utf-8-sig"
)
print("최종 shape:", clean_check.shape)
print("\nISBN 길이 분포:")
print(clean_check["ISBN"].str.len().value_counts(dropna=False))
print("\n최종 결측치:")
print(clean_check.isna().sum())

## 8. 내부·외부 분석 단위 확인

- **내부 데이터 한 행의 의미:** 베스트셀러 목록의 도서/상품 1권(원본 순위 포함).
- **외부 데이터 한 행의 의미:** 해당 상품번호 상세 페이지에서 수집한 도서별 지표 1행.
- **내부 시간 단위:** 수집 당시의 베스트셀러 순위 스냅샷. 정확한 기준 기간 **미확인**.
- **외부 시간 단위:** 상세 페이지를 읽은 시점의 값. 정확한 항목별 UTC 기록 **없음**.
- **Timezone:** 시계열 병합 없음(도서 식별자 기준); 수집 시각 기록 시 UTC를 사용하도록 개선 필요.
- **내부 지역 단위:** YES24 대한민국 온라인 도서 서비스 목록.
- **외부 지역 단위:** 같은 YES24 도서 상세 페이지(지역 단위 집계 없음).
- **연결 Key:** `상품번호`.
- **기대 병합 관계:** `one_to_one`.

### 단위를 맞추기 위해 수행한 집계/정규화
한 행이 상품 하나인 상태를 유지했으므로 월별·지역별 집계는 하지 않았다. `상품번호`를 문자열로 변환하고 `".0"` 꼬리표/공백을 제거하여 Key 타입을 통일했다. **랭킹 시점과 상세 페이지 조회 시점이 완전히 같지는 않을 수 있다는 한계**는 남는다.

## 9. 내부 데이터와 병합 검증

- **Join Key:** `상품번호`.
- **`validate`:** `one_to_one`.
- **외부 오른쪽 Key 중복 수:** 0.
- **병합 전 내부 행 수:** 206.
- **병합 후 행 수:** 206.
- **Row Count Preserved:** `True`.
- **`left_only_count`:** 0.
- **`both_count`:** 206.
- **최종 컬럼 수:** 기존 11개 + 추가 4개 = 15개.
- **저장 파일:** `data/raw/books_raw_total.csv` (사용자가 지정한 경로; 성격상 가공/중간 데이터임).

### 미매칭 해석
미매칭은 실제 점수/판매량이 0이라는 증거가 아니며, 수집 실패·접근 제한·상품번호 불일치·삭제된 상품 등으로 발생할 수 있다. 따라서 0으로 채우지 않고 별도 검토해야 한다.

```text
실제 값 0 ≠ 외부 데이터 부재
```

> 병합 화면 캡처: `chapter13/images/step07_merge.png` (직접 저장 시 증빙됨).

In [ ]:
raw_check = pd.read_excel(RAW_PATH, dtype={"ISBN": "string", "상품번호": "string"})
raw_check["상품번호"] = raw_check["상품번호"].str.strip().str.replace(r"\.0$", "", regex=True)
numeric_view["상품번호"] = numeric_view["상품번호"].str.strip().str.replace(r"\.0$", "", regex=True)

merged_check = raw_check.merge(
    numeric_view[required_ext],
    on="상품번호", how="left", validate="one_to_one", indicator=True
)
merge_counts = merged_check["_merge"].value_counts()
merge_evidence = {
    "right_key_duplicate_count": int(numeric_view["상품번호"].duplicated().sum()),
    "before_rows": len(raw_check),
    "after_rows": len(merged_check),
    "row_count_preserved": len(raw_check) == len(merged_check),
    "left_only_count": int(merge_counts.get("left_only", 0)),
    "both_count": int(merge_counts.get("both", 0)),
    "columns_excluding_indicator": len(merged_check.columns) - 1,
}
display(pd.DataFrame([merge_evidence]))
assert merge_evidence["before_rows"] == 206
assert merge_evidence["after_rows"] == 206
assert merge_evidence["left_only_count"] == 0
assert merge_evidence["both_count"] == 206
assert merge_evidence["columns_excluding_indicator"] == 15
print("병합 검증: PASS")

## 10. 검색 API / 웹 결과의 대표성

- **Query:** 검색 API 사용 안 함(**해당 없음**); 상품번호로 상세 URL에 직접 접근.
- **Sort:** 외부 API 검색 정렬 없음. 내부 Excel 순위를 유지.
- **Page / Start / Page Size:** 검색 API 페이지네이션 **해당 없음**.
- **Request Count:** 원본 고유 상품번호 **206개**가 대상이었으나, 소규모 시험·재실행·중단/재개·Selenium 조회가 있어 **실제 HTTP 요청 총 횟수는 기록되지 않음**.
- **Collected At UTC:** 미기록.
- **검색 서비스/플랫폼:** YES24 공식 도서 상세 페이지/베스트셀러 목록.
- **반환 범위 한계:** 사용자가 보유한 206권 목록에 한정; YES24 전체 판매도서·국내 출판시장 모집단 아님.
- **중복 처리:** 상품번호 기준 고유 206개, 중복 0.

### 잘못된 과대 해석 문장
“YES24 베스트셀러 206권의 평균별점과 판매지수 관계는 대한민국 모든 책의 판매 행동을 대표한다.”

### 범위를 명확히 한 수정 문장
“**이 분석에서 사용한 YES24 베스트셀러 206개 상품의 수집 시점 지표**를 기준으로 순위·별점·리뷰건수·판매지수 간 연관성을 탐색할 수 있다.”

### 대표성 한계
베스트셀러로 선정된 상품에 편중되었으며, 순위의 기준 기간·상세 지표 조회 시점도 충분히 기록되지 않았다. 리뷰 참여자는 구매자 전체와 같지 않고, 판매지수는 실제 판매 권수가 아니다.

## 11. 공개 HTML 수집 검토

- **공식 파일/API 존재 여부 확인:** 공식 API/다운로드와 HTML을 체계적으로 비교한 근거 문서는 **미확보**. 실제 실습은 상품 상세 HTML을 선택했다.
- **robots.txt 상태:** 이전 실습 대화에서 일반 상품 경로에 명시적인 차단이 없다는 안내를 받았으나 **robots.txt 원문·캡처를 본 Notebook에 보존하지 못함**. 다음 실행 전 재검증 필요.
- **이용약관 검토:** 사용자가 열람했고, 제시된 조항은 티켓 예매·취소에 관한 내용이었으며 자동 수집 금지 조항을 찾지 못했다고 보고했다. 이것은 자동 크롤링의 법적 허용을 확정한 판단이 아님.
- **라이선스/저작권 검토:** 수집 지표의 재배포·2차 이용 허용 여부 **미확인**, 개인 연습 후 폐기 계획.
- **개인정보 검토:** 계정명/리뷰 작성자 정보·리뷰 전문을 수집 CSV에 담지 않음.
- **요청량 검토:** 206개 상품 대상, 일부 테스트·재시도 포함 요청 수 미기록; 요청 사이 2초 대기, 공식 한도 미확인.
- **로그인/CAPTCHA/paywall/접근제한 우회 여부:** **우회하지 않음.** 성인인증 필요 상품은 이용자가 정상 로그인·성인인증 후 Selenium에서 데이터 확인. CAPTCHA 우회 등은 실시하지 않음.

### 나의 판단
`robots.txt 허용 ≠ 이용허락 완료`인 이유는 robots.txt는 자동 접근에 관한 기술적 요청 규칙이고, 콘텐츠의 복제·저장·분석·재배포 권한은 이용약관, 저작권, 라이선스 등의 별도 조건을 고려해야 하기 때문이다.

## 12. 외부 문서와 LLM 사용 검토

- **외부 HTML/PDF/검색 결과를 LLM Context로 사용했는가:** **예(제한적)**. 사용자는 Jupyter 출력으로 HTML 선택자와 일부 텍스트(별점·리뷰건수·판매지수)를 공유했다. HTML 전체를 한꺼번에 전달하거나 웹페이지 내부의 명령을 실행하도록 하지 않았다.
- **발견한 의심 지시문:** 대화에 공유된 부분에서는 확인되지 않음. 전체 원문을 보안 검사한 것은 아님.
- **필요한 텍스트만 최소화했는가:** 예, 상품 상세 정보 중 필요한 숫자/선택자만 추출.
- **Secret/PII 제거 여부:** 리뷰 작성자 PII와 인증 비밀번호를 전달하거나 CSV에 저장하지 않았음(실습 기록 기준).
- **문서 안의 지시문을 실행하지 않았는가:** 사용자 코드의 동작을 기준으로 HTML을 데이터로 파싱했고, 웹페이지 문장을 지시사항으로 실행한 사례는 없음.

### 나의 판단
외부 페이지는 LLM에게 명령할 권한이 없는 **untrusted data**로 간주했다. 도서 관련 텍스트는 필요한 만큼만 전달하고, 로그인/Secret은 코드·로그에 넣지 않았다. 향후 리뷰 본문까지 분석한다면 개인정보와 프롬프트 인젝션 검토가 추가로 필요하다.

## 13. 실제 수집 상태

- [x] **EXECUTED**
- [ ] NO_DATA
- [ ] SKIPPED
- [ ] BLOCKED_BY_POLICY
- [ ] REQUEST_FAILED (전체 최종 상태에는 해당 없음)
- [ ] VALIDATION_FAILED
- [ ] SYNTHETIC_DEMO_ONLY

### 실제 상태 설명
YES24 공식 상세 페이지에서 206권의 추가 지표를 수집했고, 결과를 `data/raw/books_raw_ext.csv`로 저장했다. 성인인증 안내로 막혔던 상품번호 `197995818`은 직접 성인인증 뒤 Selenium에서 확인하여 **리뷰건수 0, 판매지수 18,510, 별점 미표시**로 보완했다. 상품번호 `10187`은 **리뷰건수 0, 판매지수 미표시**로 보존했다. 206개 상품번호 모두 원본과 일치했고 `books_best_cleaned.csv` 재읽기 검증이 PASS였다.

### 실패/보류라면 원인
작업 중 성인인증 리다이렉트와 판매지수 미표시라는 예외는 있었지만, 전체 수집 작업은 완료했다. **1건의 판매지수 결측은 실제 미제공으로 유지**했다. 정확한 수집 UTC, 정책 증빙, 최초 스냅샷의 재현성은 별도 보완 사항이다.

### Demo/Synthetic 데이터를 사용했다면
**사용하지 않음.** 제시된 집계는 실제 사용자가 수집·검증한 YES24 데이터에서 나온 결과다.

## 14. 외부 데이터가 준 추가 맥락

### 결과 관찰
- 외부 수집으로 도서별 평균별점, 리뷰총점, 리뷰건수, 판매지수 4개 지표를 추가해 **11개 → 15개 컬럼**으로 확장했다.
- 리뷰총점이 누락된 **19권은 모두 리뷰건수 0권**이었다.
- 평균별점 결측은 **13권**으로, 리뷰총점 결측(19권)과 개수가 다르다. 두 평가 항목은 동일한 지표로 취급하지 않는다.
- 판매지수는 **205권에서 확인**, 마지막 1권은 확인되지 않았다.
- **순위와 외부 지표의 통계적 상관 정도는 아직 계산/해석하지 않았다.** 아래 코드 셀을 실행한 결과로 추가할 수 있다.

### 나의 해석과 판단
순위만으로는 알 수 없었던 평가 참여 정도(리뷰건수), 평가 점수, 플랫폼 판매지수를 도서 단위로 비교할 수 있게 됐다. 그러나 평균별점과 리뷰총점은 서로 다르며, 리뷰건수가 적은 고평점 도서에서는 점수 안정성이 낮을 수 있어 함께 살펴봐야 한다.

### 업무·분석적 의미
순위가 비슷한 도서 간 판매지수·리뷰건수 차이, 관리분류별 분포, 평가 점수와 참여 규모의 차이를 확인하는 탐색 분석에 활용할 수 있다.

### 인과 해석 한계
별점이 높고 판매지수가 높더라도 “높은 별점이 판매를 증가시켰다”고 확정할 수 없다. 인기 도서에 리뷰가 더 붙는 역인과, 출간 시기·인지도·장르 등 교란 변수가 있고, 현재는 단일 시점에 가까운 관찰 자료다.

### 대표성 / 수집 시점 / 정책 한계
206권은 특정 YES24 베스트셀러 집합이지 전체 도서 시장이 아니다. 순위 집계 기준 시점과 개별 상세 페이지 조회 시점이 정확히 일치하는지도 확인되지 않았다. 실제 수집된 데이터를 재배포하려면 별도 이용조건 검토가 필요하다.

In [ ]:
# 결과 관찰 확장용 탐색 분석: 실행 결과를 보고 해석을 Markdown에 추가
analysis_df = clean_check.copy()
for col in ["순위", "평균별점", "리뷰총점", "리뷰건수", "판매지수"]:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

print("분석 대상 도서:", len(analysis_df))
print("\n외부 지표 기술통계:")
display(analysis_df[["평균별점", "리뷰총점", "리뷰건수", "판매지수"]].describe())

print("\n순위와 외부 지표의 Spearman 순위상관 (인과관계 아님):")
for col in ["평균별점", "리뷰총점", "리뷰건수", "판매지수"]:
    corr = analysis_df[["순위", col]].dropna().corr(method="spearman").iloc[0, 1]
    n = len(analysis_df[["순위", col]].dropna())
    print(f"{col}: rho={corr:.3f}, 유효 표본={n}")

## 15. 준비 Evidence 파일 확인

아래 10개 파일은 **수업용 `scripts/run_external_data_collection.py` 준비 Runner가 생성하도록 안내된 산출물**이다. **지금까지의 대화만으로 생성 완료 여부는 알 수 없다.** 체크박스는 실제 생성 여부를 다음 코드로 확인한 뒤 표시한다.

- [ ] `ch13_external_data_plan.csv`
- [ ] `ch13_collection_method_summary.csv`
- [ ] `ch13_external_integration_plan.csv`
- [ ] `ch13_external_data_checklist.csv`
- [ ] `ch13_external_data_log.csv`
- [ ] `ch13_env_key_status.csv`
- [ ] `ch13_collection_metadata_template.csv`
- [ ] `ch13_api_code_review_checklist.csv`
- [ ] `ch13_network_execution_gate.csv`
- [ ] `ch13_external_data_summary.md`

### 가장 중요한 Evidence 3개와 이유
1. **`ch13_external_data_log.csv`**: 어떤 데이터 수집이 어떤 상태로 끝났는지, 실패/미실행이 있었는지 구분한다. (기본 템플릿의 로그는 실제 YES24 크롤링 로그가 아니므로 직접 갱신 필요.)
2. **`ch13_network_execution_gate.csv`**: 수집 전 출처·정책·요청 범위 승인 과정을 명시한다. (준비 기본 OFF와 이미 실행한 YES24 요청의 상태를 혼동하지 않는다.)
3. **수집 CSV의 SHA-256을 포함하는 Metadata JSON**: 원본 내용·출처·현재 보존본을 추적한다. (Section 5에서 사후 생성 가능.)

### Runner 사용 시 주의
원래 수업 Runner는 **네트워크 요청을 하지 않는다**. 아래 첫 셀은 파일 존재만 확인한다. 필요하다면 둘째 셀에서 `RUN_SETUP = True`로 바꿔 실행하며, 생성된 템플릿을 실제 수집 이력으로 가장하지 않는다.

In [4]:
expected_reports = [
    "ch13_external_data_plan.csv",
    "ch13_collection_method_summary.csv",
    "ch13_external_integration_plan.csv",
    "ch13_external_data_checklist.csv",
    "ch13_external_data_log.csv",
    "ch13_env_key_status.csv",
    "ch13_collection_metadata_template.csv",
    "ch13_api_code_review_checklist.csv",
    "ch13_network_execution_gate.csv",
    "ch13_external_data_summary.md",
]
report_status = pd.DataFrame([
    {"Evidence": name, "exists": (REPORT_DIR / name).is_file()}
    for name in expected_reports
])
display(report_status)
print("확인된 Evidence:", int(report_status["exists"].sum()), "/", len(report_status))

,Evidence,exists
0,ch13_external_data_plan.csv,False
1,ch13_collection_method_summary.csv,False
2,ch13_external_integration_plan.csv,False
3,ch13_external_data_checklist.csv,False
4,ch13_external_data_log.csv,False
5,ch13_env_key_status.csv,False
6,ch13_collection_metadata_template.csv,False
7,ch13_api_code_review_checklist.csv,False
8,ch13_network_execution_gate.csv,False
9,ch13_external_data_summary.md,False


확인된 Evidence: 0 / 10


In [ ]:
# 선택 사항: 수업용 준비 Runner를 실행해 기본 템플릿 생성 (네트워크 요청 0회)
# 이미 존재하는 파일에 미치는 영향은 src 구현에 따라 다를 수 있으므로 실행 전 확인.
RUN_SETUP = False
if RUN_SETUP:
    import subprocess
    import sys
    runner = ROOT / "scripts/run_external_data_collection.py"
    if not runner.exists():
        raise FileNotFoundError(runner)
    subprocess.run([sys.executable, str(runner)], cwd=ROOT, check=True)
    print("준비 Runner 실행 완료. 실제 수집 이력은 별도 기록해야 합니다.")
else:
    print("Runner 미실행: RUN_SETUP=True로 바꾸면 실행할 수 있습니다.")

## 16. 최종 사용 판단

- [ ] 현재 분석에 무조건 사용 가능
- [x] **추가 검증 후 사용 가능**
- [ ] 현재 사용 보류

### 판단 근거
1. **수집/병합 구조는 PASS:** 206권, 중복 0, Key 미매칭 0, 15개 컬럼 유지, 별점/리뷰건수/판매지수 범위 검증 통과.
2. **결측 의미는 보존:** 리뷰 0건과 리뷰총점 결측 19건이 일치하며, 판매지수 미제공 1건을 임의로 0으로 대체하지 않음.
3. **운영 Evidence는 보완 필요:** 순위 기준일, 정확한 수집 UTC, 공식 문서/이용약관 캡처, 수집 전 불변 Raw 스냅샷과 Gate 승인 로그가 완전하지 않음.

### 신뢰할 수 있는 범위
현재 확보한 **YES24 베스트셀러 206권의 상품 단위 지표**를 이용한 기술통계와 탐색적 비교는 구조 검증을 거쳐 수행할 수 있다. 단, 별점 간 의미 차이와 각 지표의 미제공 값을 구분해야 한다.

### 현재 결과만으로 말할 수 없는 것
전체 도서 시장의 경향, 실제 도서 판매량, 별점이 매출을 증가시켰다는 인과 효과, 지표의 장기 변화, 약관상 제3자 재배포가 허용된다는 사실.

### 다음 수집 시 다시 확인할 것
1. 베스트셀러 **기준 기간과 수집 UTC를 요청마다 기록**하고 공개 문서/약관·robots를 캡처.
2. 네트워크 Gate·공식 rate limit·개별 timeout·일시 오류 정책·접근 인증 후 리다이렉트 검증.
3. Raw를 생성 시점부터 **timestamped immutable snapshot**으로 저장하고 파일 해시·출처·재배포/삭제 정책을 Metadata에 연결.

---
## 최종 체크

- [x] 외부 데이터 필요성을 질문에서 정의했습니다.
- [ ] 공식 파일/API를 우선 확인했다는 근거를 보존했습니다.
- [ ] 실행 시점의 공식 문서와 이용조건을 증빙으로 저장했습니다.
- [ ] Data Reference Date와 Collected At의 **실제 값**을 구분하여 기록했습니다.
- [ ] 수업 준비 Runner의 Network Gate 기본 OFF를 실행 결과로 확인했습니다.
- [x] 인증 비밀번호/API Secret 실제 값을 Notebook 코드·수집 CSV에 넣지 않았습니다(사용자 작업 기록 기준).
- [ ] 로컬 환경에서 MISSING / PLACEHOLDER / CONFIGURED를 실행 결과로 확인했습니다.
- [x] timeout / retry / rate limit / pagination의 차이를 문서화했습니다.
- [x] HTTP 성공과 업무 성공을 구분했습니다.
- [x] 원본/외부 수집/병합/최종 파일을 구별하여 저장했습니다.
- [ ] **수집 시작 시점부터** Raw Snapshot을 비덮어쓰기 방식으로 보존했습니다.
- [x] SHA-256을 데이터 품질 보증으로 오해하지 않았습니다.
- [x] 외부 데이터 품질과 결측을 검증했습니다.
- [x] 내부·외부 분석 단위를 맞췄습니다.
- [x] 외부 오른쪽 Key와 병합 미매칭을 검증했습니다.
- [x] 검색/웹 결과를 전체 모집단으로 과장하지 않았습니다.
- [x] 외부 HTML을 untrusted data로 취급했습니다.
- [x] 실제 수집 실패/미제공 값을 가짜 결과로 대체하지 않았습니다.
- [x] 인과관계를 과대 해석하지 않았습니다.
- [ ] 최종 Notebook URL을 제출했습니다.

### 마무리
**데이터 처리 결과:** `data/processed/books_best_cleaned.csv` — 206행 × 15열, `상품번호` 유일, 병합 100% 성공(사용자 제공 실행 결과).  
**제출 전에:** 코드 셀을 위에서부터 실행하고, 미확인 항목을 사실대로 보완하며, 실제 캡처·Evidence 파일·GitHub URL을 첨부한다.  
**개인 학습 완료 후:** 약속한 보존/삭제 정책에 따라 원본·스냅샷·가공본을 검토 후 폐기한다.